### Olmo-3 Architecture

In [14]:
## models from allenAI (olmo series)
# USE_MODEL = "Olmo-3-1025-7B"
# USE_MODEL = "Olmo-3-1125-32B"
USE_MODEL = "Olmo-3-7B-Instruct"
# USE_MODEL = "Olmo-3-32B-Instruct"
# USE_MODEL = "Olmo-3-7B-Think"
# USE_MODEL = "Olmo-3-32B-Think"
# USE_MODEL = "Olmo-3-7B-RLZero-IF"

In [15]:
## Architecture  
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # cfg:
        # emb_dim    = d_model
        # hidden_dim = d_ff (typically ~4 * d_model or 8/3 * d_model for SwiGLU)
        # dtype      = torch.float16 / bfloat16 / float32

        self.fc1 = nn.Linear(
            cfg['emb_dim'], 
            cfg['hidden_dim'], 
            dtype=cfg["dtype"], 
            bias=False
        )  # (d_model → d_ff)

        self.fc2 = nn.Linear(
            cfg['emb_dim'], 
            cfg['hidden_dim'], 
            dtype=cfg['dtype'], 
            bias=False
        )  # (d_model → d_ff)  <-- gating branch

        self.fc3 = nn.Linear(
            cfg['hidden_dim'], 
            cfg['emb_dim'], 
            dtype=cfg['dtype'], 
            bias=False
        )  # (d_ff → d_model)

    def forward(self, x): 
        """
        x: (batch_size, seq_len, emb_dim)
           = (B, T, d_model)
        """

        # -------- First projections --------
        x_fc1 = self.fc1(x)
        # (B, T, d_model) → (B, T, d_ff)

        x_fc2 = self.fc2(x)
        # (B, T, d_model) → (B, T, d_ff)

        # -------- SwiGLU activation --------
        # silu(x_fc1): (B, T, d_ff)
        # elementwise multiply with x_fc2
        x = F.silu(x_fc1) * x_fc2
        # (B, T, d_ff) ⊙ (B, T, d_ff) → (B, T, d_ff)

        # -------- Projection back --------
        out = self.fc3(x)
        # (B, T, d_ff) → (B, T, d_model)

        return out
        # final: (B, T, d_model)

In [16]:
import math
import torch

def compute_rope_parameters(
    head_dim,
    theta_base=10000,
    context_length=4096,
    attention_factor=1.0,
    rope_type="default",
    rope_factor=1.0,
    rope_orig_max=8192,
    beta_fast=32.0,
    beta_slow=1.0,
    dtype=torch.float32,
):
    """
    Compute RoPE (Rotary Positional Embedding) parameters.

    Goal:
        Precompute cos/sin tables used to rotate Q/K vectors.

    Returns:
        cos, sin: [context_length, head_dim]
            - each row = one position
            - each column = one dimension (paired internally)
    """

    # --- Basic constraints ---
    # head_dim must be even because RoPE operates on pairs (x1, x2)
    assert head_dim % 2 == 0

    # number of 2D rotation planes (pairs)
    # e.g. D=64 → 32 independent rotations
    num_freqs = head_dim // 2   # scalar

    # ============================================================
    # =============== YARN (Scaled RoPE Variant) ==================
    # ============================================================
    if rope_type == "yarn":

        def find_correction_dim(num_rotations, dim, base, max_position_embeddings):
            """
            Map a rotation threshold → frequency index.

            Intuition:
                Higher frequency → faster rotation across positions.

            This finds:
                which dimension index corresponds to a certain rotation speed.

            Returns:
                float index in [0, dim)
            """
            return (
                dim
                * math.log(max_position_embeddings / (num_rotations * 2 * math.pi))
                / (2 * math.log(base))
            )

        def find_correction_range(low_rot, high_rot, dim, base, max_position_embeddings):
            """
            Find frequency index range where we transition:
                extrapolation → interpolation

            Returns:
                (low_idx, high_idx)  both ints
            """
            low = find_correction_dim(low_rot, dim, base, max_position_embeddings)
            high = find_correction_dim(high_rot, dim, base, max_position_embeddings)

            low = math.floor(low)
            high = math.ceil(high)

            # clamp to valid dimension range
            return max(low, 0), min(high, dim - 1)

        def linear_ramp_factor(low, high, dim):
            """
            Build blending weights across frequency indices.

            Output:
                ramp: [num_freqs]

            Behavior:
                index < low  → 0
                index > high → 1
                between      → linear interpolation
            """
            if low == high:
                high += 1e-6

            # [num_freqs]
            x = torch.arange(dim, dtype=torch.float32)

            # normalize into [0,1] range
            ramp = (x - low) / (high - low)

            # clamp to [0,1]
            return torch.clamp(ramp, 0, 1)

        # --------------------------------------------------------
        # Base frequencies (log-spaced across dimensions)
        # shape: [num_freqs]
        # --------------------------------------------------------
        pos_freqs = theta_base ** (
            torch.arange(0, head_dim, 2, dtype=dtype) / head_dim
        )
        # NOTE:
        # torch.arange(0, head_dim, 2) → [0,2,4,...] → num_freqs elements

        # Two variants of inverse frequency
        # both: [num_freqs]
        inv_freq_extrapolation = 1.0 / pos_freqs
        inv_freq_interpolation = 1.0 / (rope_factor * pos_freqs)

        # --------------------------------------------------------
        # Find where to transition between the two regimes
        # --------------------------------------------------------
        low, high = find_correction_range(
            beta_fast, beta_slow, num_freqs, theta_base, rope_orig_max
        )

        # --------------------------------------------------------
        # Build blending ramp across dimensions
        # shape: [num_freqs]
        # --------------------------------------------------------
        ramp = linear_ramp_factor(low, high, num_freqs).to(dtype=dtype)

        # --------------------------------------------------------
        # Blend frequencies per dimension
        # shape: [num_freqs]
        # --------------------------------------------------------
        inv_freq = (
            inv_freq_interpolation * ramp
            + inv_freq_extrapolation * (1 - ramp)
        )

    # ============================================================
    # =================== DEFAULT RoPE ============================
    # ============================================================
    else:
        # Standard inverse frequencies (log-spaced)
        # shape: [num_freqs]
        inv_freq = 1.0 / (
            theta_base
            ** (
                torch.arange(0, head_dim, 2, dtype=dtype)[:num_freqs].float()
                / head_dim
            )
        )

    # ============================================================
    # ==================== BUILD ANGLES ===========================
    # ============================================================

    # Positions along sequence
    # shape: [T]
    positions = torch.arange(context_length, dtype=dtype)

    # Outer product:
    # positions[:, None] * inv_freq[None, :]
    #
    # shape:
    #   [T, num_freqs]
    #
    # Each entry:
    #   angles[t, i] = position[t] * frequency[i]
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)

    # ============================================================
    # Expand to full head_dim
    # ============================================================

    # Duplicate each frequency to match (x1, x2) pairing
    #
    # before: [T, num_freqs]
    # after : [T, head_dim]
    #
    # layout becomes:
    #   [θ0, θ1, ..., θ31 | θ0, θ1, ..., θ31]
    angles = torch.cat([angles, angles], dim=1)

    # ============================================================
    # Compute cos/sin tables
    # ============================================================

    # shape: [T, D]
    cos = torch.cos(angles) * attention_factor
    sin = torch.sin(angles) * attention_factor

    # These will be used later as:
    #   x_rot = x*cos + rotated*sin

    return cos, sin

In [17]:
def apply_rope(x, cos, sin, offset=0):
    """
    Apply Rotary Positional Embedding (RoPE) to input tensor.

    Args:
        x   : [B, H, T, D]
              - B: batch size
              - H: number of attention heads
              - T: sequence length (current chunk)
              - D: head_dim (must be even)

        cos : [context_len, D]
        sin : [context_len, D]

        offset : int
            - starting position index (important for KV-cache / decoding)

    Returns:
        x_rotated : [B, H, T, D]
    """

    # ============================================================
    # ====================== SHAPES ===============================
    # ============================================================

    # unpack dimensions
    # x: [B, H, T, D]
    B, H, T, D = x.shape

    # RoPE requires pairs → D must be even
    assert D % 2 == 0


    # ============================================================
    # ============== SPLIT INTO ROTATION PAIRS ====================
    # ============================================================

    # Split last dimension into two halves:
    # x = [x1 | x2]
    #
    # shapes:
    #   x1: [B, H, T, D/2]
    #   x2: [B, H, T, D/2]
    #
    # Interpretation:
    #   pair i = (x1[..., i], x2[..., i])
    #
    # Example:
    #   D=64 → 32 independent 2D rotations
    x1 = x[..., : D // 2]
    x2 = x[..., D // 2 :]


    # ============================================================
    # ============== SELECT POSITIONAL ANGLES =====================
    # ============================================================

    # cos/sin originally:
    #   [context_len, D]
    #
    # We select only the rows corresponding to current tokens:
    #
    # rows: offset → offset + T - 1
    #
    # result:
    #   cos: [T, D]
    #   sin: [T, D]
    #
    # Meaning:
    #   each token position t gets its own rotation angles
    #
    # Example:
    #   offset=5, T=2 → use positions 5 and 6
    cos = cos[offset : offset + T, :]
    sin = sin[offset : offset + T, :]


    # ============================================================
    # ============== PREPARE FOR BROADCASTING =====================
    # ============================================================

    # Expand to match x shape for elementwise ops
    #
    # before:
    #   cos: [T, D]
    #
    # after:
    #   cos: [1, 1, T, D]
    #
    # This allows broadcasting across:
    #   batch (B) and heads (H)
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)


    # ============================================================
    # ============== BUILD PERPENDICULAR VECTOR ===================
    # ============================================================

    # Construct 90° rotated version of x:
    #
    # pair (x1, x2) → (-x2, x1)
    #
    # shapes:
    #   x1: [B,H,T,D/2]
    #   x2: [B,H,T,D/2]
    #
    # result:
    #   rotated: [B,H,T,D]
    #
    # Interpretation:
    #   this is the "perpendicular direction" in 2D rotation
    rotated = torch.cat((-x2, x1), dim=-1)


    # ============================================================
    # ================= APPLY ROTATION ============================
    # ============================================================

    # Elementwise:
    #
    # x_rot = x * cos + rotated * sin
    #
    # For each:
    #   (b, h, t, d):
    #       x_rot[b,h,t,d] =
    #           x[b,h,t,d] * cos[t,d]
    #         + rotated[b,h,t,d] * sin[t,d]
    #
    # In pair form:
    #
    #   x1' = x1*cos - x2*sin
    #   x2' = x2*cos + x1*sin
    #
    # Meaning:
    #   rotate each (x1[i], x2[i]) by angle θ(t, i)
    #
    # where:
    #   θ depends on:
    #       position t
    #       pair index i (frequency)
    x_rotated = (x * cos) + (rotated * sin)


    # ============================================================
    # =================== DTYPE SAFETY ============================
    # ============================================================

    # Ensure output dtype matches input (important for fp16/bf16)
    return x_rotated.to(dtype=x.dtype)

In [18]:
## RMSNorm 

import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.eps = eps
        # weight: [D]  (per-feature scaling)
        self.weight = nn.Parameter(torch.ones(dim))


    def forward(self, x):
        """
        x: [B, T, D]
           B = batch size
           T = sequence length (e.g., 14 tokens)
           D = embedding dim (e.g., 64)
        """

        input_dtype = x.dtype

        # ---- Step 1: cast to float32 for numerical stability ----
        x_f = x.float()
        # x_f: [B, T, D]

        # ---- Step 2: compute mean square (RMS denominator) ----
        var = x_f.pow(2).mean(dim=-1, keepdim=True)
        """
        x_f.pow(2): [B, T, D]
        mean over last dim (D):

        var: [B, T, 1]

        Each token now has ONE scalar = mean(x_i^2)
        """

        # ---- Step 3: normalize ----
        x_norm = x_f * torch.rsqrt(var + self.eps)
        """
        rsqrt(var): [B, T, 1]

        broadcast multiply:
        [B, T, D] * [B, T, 1] → [B, T, D]

        x_norm: [B, T, D]
        """

        # ---- Step 4: scale with learned weight ----
        out = self.weight * x_norm
        """
        weight: [D]

        broadcast:
        [D] → [1, 1, D]

        [B, T, D] * [1, 1, D] → [B, T, D]
        """

        # ---- Step 5: cast back to original dtype ----
        return out.to(input_dtype)
        # final output: [B, T, D]

In [19]:
import torch
import torch.nn as nn

class GroupQueryAttention(nn.Module):

    def __init__(
        self,
        d_in,                # int: model dim
        num_heads,           # int
        num_kv_groups,       # int
        head_dim,            # int
        attention_bias=False,
        dtype=None,
        sliding_window=None,
        attn_type="full_attention"
    ):
        super().__init__()

        assert num_heads % num_kv_groups == 0, \
            "num_heads must be divisible by num_kv_groups"

        # -----------------------------
        # Core hyperparameters
        # -----------------------------
        self.num_heads = num_heads                      # e.g. 32
        self.num_kv_groups = num_kv_groups              # e.g. 8
        self.group_size = num_heads // num_kv_groups    # e.g. 4

        self.head_dim = head_dim                        # e.g. 128

        # Total Q projection dim
        self.d_out = num_heads * head_dim               # (32 * 128 = 4096)

        self.attn_type = attn_type
        self.sliding_window = sliding_window if attn_type == "sliding_window" else None


        # -----------------------------
        # PROJECTIONS (VERY IMPORTANT)
        # -----------------------------

        # Input:  (b, T, d_in)
        # Output: (b, T, num_heads * head_dim)
        self.W_query = nn.Linear(d_in, self.d_out, bias=attention_bias, dtype=dtype)
        # Weight shape:
        # (d_in, num_heads * head_dim)


        # Input:  (b, T, d_in)
        # Output: (b, T, num_kv_groups * head_dim)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=attention_bias, dtype=dtype)
        # Weight shape:
        # (d_in, num_kv_groups * head_dim)


        # Same as keys
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=attention_bias,dtype=dtype)
        # Weight shape:
        # (d_in, num_kv_groups * head_dim)


        # Output projection
        # Input:  (b, T, num_heads * head_dim)
        # Output: (b, T, d_in)
        self.out_proj = nn.Linear(self.d_out, d_in, bias=attention_bias, dtype=dtype)
        # Weight shape:
        # (num_heads * head_dim, d_in)


        # -----------------------------
        # RMSNorm (over flattened dim)
        # -----------------------------

        # Input:  (b, T, num_heads * head_dim)
        # Output: same
        self.q_norm = RMSNorm(self.d_out)
        # Learnable weight:
        # (num_heads * head_dim,)


        # Input:  (b, T, num_kv_groups * head_dim)
        self.k_norm = RMSNorm(num_kv_groups * head_dim)
        # Learnable weight:
        # (num_kv_groups * head_dim,)





    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        # x: (b, num_tokens, d_in)

        b, num_tokens, _ = x.shape
        # b = batch size
        # num_tokens = sequence length (current chunk)

        # --------------------------------------------------
        # Apply projections
        # --------------------------------------------------

        queries = self.W_query(x)
        # (b, num_tokens, num_heads * head_dim)

        keys = self.W_key(x)
        # (b, num_tokens, num_kv_groups * head_dim)

        values = self.W_value(x)
        # (b, num_tokens, num_kv_groups * head_dim)

        # --------------------------------------------------
        # Normalize q and k (no shape change)
        # --------------------------------------------------

        queries = self.q_norm(queries)
        # (b, num_tokens, num_heads * head_dim)

        keys_new = self.k_norm(keys)
        # (b, num_tokens, num_kv_groups * head_dim)

        # --------------------------------------------------
        # Reshape into heads
        # --------------------------------------------------

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # (b, num_tokens, num_heads, head_dim)

        queries = queries.transpose(1, 2)
        # (b, num_heads, num_tokens, head_dim)

        # --------------------------------------------------

        keys_new = keys_new.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        # (b, num_tokens, num_kv_groups, head_dim)

        keys_new = keys_new.transpose(1, 2)
        # (b, num_kv_groups, num_tokens, head_dim)

        # --------------------------------------------------

        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        # (b, num_tokens, num_kv_groups, head_dim)

        values_new = values_new.transpose(1, 2)
        # (b, num_kv_groups, num_tokens, head_dim)



       ## cache un-rotated KV

        prev_len = 0
        # prev_len: int
        # number of tokens stored in cache (sequence length of prev_k)

        if cache is not None:
            # cache = (prev_k, prev_v)

            prev_k, prev_v = cache
            # prev_k: (b, num_kv_groups, prev_len, head_dim)
            # prev_v: (b, num_kv_groups, prev_len, head_dim)

            if prev_k is not None:

                prev_len = prev_k.size(2)
                # prev_len: int
                # = sequence length of cached tokens

                keys_cat_raw = torch.cat([prev_k, keys_new], dim=2)
                # prev_k:   (b, num_kv_groups, prev_len,   head_dim)
                # keys_new: (b, num_kv_groups, num_tokens, head_dim)
                # --------------------------------------------------
                # keys_cat_raw: (b, num_kv_groups, prev_len + num_tokens, head_dim)

                values_cat_raw = torch.cat([prev_v, values_new], dim=2)
                # prev_v:     (b, num_kv_groups, prev_len,   head_dim)
                # values_new: (b, num_kv_groups, num_tokens, head_dim)
                # --------------------------------------------------
                # values_cat_raw: (b, num_kv_groups, prev_len + num_tokens, head_dim)

            else:
                # cache exists but empty

                keys_cat_raw = keys_new
                # (b, num_kv_groups, num_tokens, head_dim)

                values_cat_raw = values_new
                # (b, num_kv_groups, num_tokens, head_dim)

        else:
            # no cache at all (first forward pass)

            keys_cat_raw = keys_new
            # (b, num_kv_groups, num_tokens, head_dim)

            values_cat_raw = values_new
            # (b, num_kv_groups, num_tokens, head_dim)




                ## ------------------------------------------------------------
        ## Apply RoPE with offsets (handles KV cache correctly)
        ## ------------------------------------------------------------

        # queries: (b, num_heads, num_tokens, head_dim)
        # cos/sin: (max_seq_len, head_dim) or broadcastable to queries
        # start_pos: scalar → absolute position of first new token (e.g., t)

        # After RoPE:
        # queries: (b, num_heads, num_tokens, head_dim)
        # positions applied = [start_pos, ..., start_pos + num_tokens - 1]
        queries = apply_rope(queries, cos, sin, offset=start_pos)


        # keys_cat_raw: (b, num_kv_groups, total_seq_len, head_dim)
        # where:
        #   total_seq_len = prev_len + num_tokens
        #   prev_len = length of cached tokens

        # offset = start_pos - prev_len ensures:
        #   cached keys → positions [0, ..., prev_len-1]
        #   new keys    → positions [prev_len, ..., start_pos + num_tokens - 1]

        # After RoPE:
        # keys: (b, num_kv_groups, total_seq_len, head_dim)
        keys = apply_rope(keys_cat_raw, cos, sin, offset=start_pos - prev_len)


        ## ------------------------------------------------------------
        ## Expand KV groups → match number of query heads (GQA)
        ## ------------------------------------------------------------

        # Before expansion:
        # queries: (b, num_heads,       num_tokens,    head_dim)
        # keys:    (b, num_kv_groups,   total_seq_len, head_dim)
        # values_cat_raw: same as keys (but no RoPE applied)

        # Relationship:
        #   num_heads = num_kv_groups * group_size

        if self.group_size > 1:

            # repeat_interleave along head dimension (dim=1)
            # This duplicates each KV head "group_size" times

            # keys:
            # (b, num_kv_groups, total_seq_len, head_dim)
            # → (b, num_kv_groups * group_size, total_seq_len, head_dim)
            # = (b, num_heads, total_seq_len, head_dim)
            keys = keys.repeat_interleave(self.group_size, dim=1)


            # values_cat_raw:
            # (b, num_kv_groups, total_seq_len, head_dim)
            # → (b, num_heads, total_seq_len, head_dim)
            values = values_cat_raw.repeat_interleave(self.group_size, dim=1)

        else:
            # No expansion needed when:
            # num_heads == num_kv_groups
            # values already aligned
            values = values_cat_raw




        ## Scale queries before attention (stabilizes dot-product)
        ## ------------------------------------------------------------

        # head_dim: int (dimension per attention head)
        # scale = 1 / sqrt(head_dim)
        # e.g., if head_dim = 64 → scale = 1/8
        scale = self.head_dim ** -0.5


        # queries: (b, num_heads, num_tokens, head_dim)
        # Multiply every element in queries by the scalar "scale"

        # After scaling:
        # queries: (b, num_heads, num_tokens, head_dim)  # shape unchanged

        # Effect:
        # - reduces magnitude of Q vectors
        # - ensures Q @ K^T does not grow too large with head_dim
        # - stabilizes softmax (prevents saturation)

        # Equivalent mathematically to:
        # (Q @ K^T) / sqrt(head_dim)

        queries = queries * scale



        ## -------------------------------------------------------------
        ## Update KV cache (store UNROTATED keys and values)
        ##
        ## cache structure:
        ##   cache = (past_keys, past_values)
        ##
        ## Shapes:
        ##   past_keys   : (b, num_kv_groups, prev_len, head_dim)
        ##   past_values : (b, num_kv_groups, prev_len, head_dim)
        ##
        ##   keys_new    : (b, num_kv_groups, new_tokens, head_dim)
        ##   values_new  : (b, num_kv_groups, new_tokens, head_dim)
        ##
        ## Goal:
        ##   Append current step's K/V to past cache along sequence dimension (dim=2)
        ##
        ## Important invariant:
        ##   - We store RAW (unrotated) K/V in cache
        ##   - NO RoPE, NO scaling applied here
        ##   - Positional encoding is applied later during attention computation
        ## -------------------------------------------------------------

        if cache is not None and cache[0] is not None:
            # Existing cache present → append new tokens

            next_cache = (
                # Concatenate along sequence dimension (time axis)
                # Result: (b, num_kv_groups, prev_len + new_tokens, head_dim)
                torch.cat([cache[0], keys_new], dim=2),

                # Same for values
                # Result: (b, num_kv_groups, prev_len + new_tokens, head_dim)
                torch.cat([cache[1], values_new], dim=2),
    )

        else:
            # First step (no previous cache)
            # Initialize cache with current tokens only

            # Shapes:
            #   (b, num_kv_groups, new_tokens, head_dim)
            next_cache = (keys_new, values_new)








        ## ---------------------- Attention Computation ----------------------

        # queries: (b, num_heads, t_q, head_dim)
        # keys:    (b, num_heads, t_k, head_dim)

        # Compute raw attention scores via dot product
        # → compares each query token with all key tokens
        # Output shape: (b, num_heads, t_q, t_k)
        attn_scores = queries @ keys.transpose(2, 3)


        # ---------------------- Masking (causal / padding) ----------------------

        # mask: (b, 1 or num_heads, t_q, t_k)
        # True → position is NOT allowed (future or padding)
        # Replace disallowed positions with -inf so softmax → 0 there
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask, -torch.inf)


        # ---------------------- Softmax (normalize over keys) ----------------------

        # Apply softmax over last dimension (t_k)
        # For each query token → distribution over all keys
        # Output shape: (b, num_heads, t_q, t_k)
        attn_weights = torch.softmax(attn_scores, dim=-1)


        # ---------------------- Weighted aggregation ----------------------

        # values: (b, num_heads, t_k, head_dim)
        # Multiply attention weights with values
        # → weighted sum of value vectors per query
        # Output: (b, num_heads, t_q, head_dim)
        context = attn_weights @ values



        # ---------------------- Merge heads ----------------------

        # transpose: (b, t_q, num_heads, head_dim)
        # reshape:   (b, t_q, num_heads * head_dim) = (b, num_tokens, d_out)
        context = context.transpose(1, 2).reshape(b, num_tokens, self.d_out)


        # ---------------------- Final projection ----------------------

        # Linear projection mixes information across heads
        # Output shape: (b, num_tokens, d_out)
        out = self.out_proj(context)


        # ---------------------- Return ----------------------

        # out:        (b, num_tokens, d_out)
        # next_cache: ((b, num_kv_heads, total_seq, head_dim), same for values)
        return out, next_cache



In [20]:
## Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, cfg, attn_type):
        super().__init__()

        self.attn_type = attn_type                          # "sliding_attention" or "global_attention"
        self.sliding_window = cfg['sliding_window']         # int: max tokens to keep in cache

        # ---------------- ATTENTION ----------------
        # Input / Output:
        # x: (B, T, D)
        # → attention → (B, T, D)
        self.att = GroupQueryAttention(
            d_in=cfg['emb_dim'],           # D
            num_heads=cfg['n_heads'],      # number of query heads
            num_kv_groups=cfg['n_kv_heads'],  # KV heads (GQA)
            head_dim=cfg['head_dim'],      # per-head dimension
            attention_bias=cfg['attention_bias'],
            dtype=cfg['dtype'],
            sliding_window=cfg['sliding_window'],
            attn_type=attn_type,
        )

        # ---------------- FEEDFORWARD ----------------
        # (B, T, D) → (B, T, D)
        self.ff = FeedForward(cfg)

        # ---------------- NORMALIZATION ----------------
        # RMSNorm keeps shape same: (B, T, D)
        self.post_attention_layernorm = RMSNorm(cfg['emb_dim'], eps=cfg["rms_norm_eps"])
        self.post_feedforward_layernorm = RMSNorm(cfg['emb_dim'], eps=cfg["rms_norm_eps"])



    def forward(self, x, mask_global, mask_local, cos, sin, start_pos=0, cache=None):
        """
        x:              (B, T, D)                ← current tokens
        mask_global:    (1, 1, T, T)             ← full causal mask
        mask_local:     (1, 1, max_seq, max_seq) ← sliding window mask (precomputed)
        cache:
            k: (B, kv_heads, T_past, head_dim)
            v: (B, kv_heads, T_past, head_dim)
        """

        # ---------------- RESIDUAL SAVE ----------------
        shortcut = x                                   # (B, T, D)

        # ============================================================
        # ---------------- MASK SELECTION LOGIC -----------------------
        # ============================================================

        if self.attn_type == "sliding_attention":

            # --------- STEP 1: get past sequence length ----------
            if cache is not None and isinstance(cache, tuple):
                prev_k, _ = cache                      # (B, kv_heads, T_past, head_dim)

                # extract past token length
                prev_len = prev_k.size(2) if prev_k is not None else 0   # scalar
            else:
                prev_len = 0                           # no cache → no past tokens


            # --------- STEP 2: total tokens (past + current) ----------
            # x.size(1) = T (current tokens)
            eff_kv_len = prev_len + x.size(1)          # scalar

            # Example:
            # prev_len = 3, current T = 2 → eff_kv_len = 5
            # total tokens = [t1 t2 t3 t4 t5]


            # --------- STEP 3: slice mask to match actual tokens ----------
            # mask_local: (1, 1, max_seq, max_seq)
            # after slicing:
            # attn_mask: (1, 1, max_seq, eff_kv_len)
            attn_mask = mask_local[..., -eff_kv_len:]

            # This ensures mask matches number of keys (tokens)


        else:
            # --------- GLOBAL ATTENTION ----------
            # mask_global: (1, 1, T, T)
            attn_mask = mask_global



        # ============================================================
        # ---------------- ATTENTION COMPUTATION ----------------------
        # ============================================================

        # Input:
        # x: (B, T, D)
        # attn_mask: (1, 1, ?, eff_kv_len)
        # cache used inside attention

        x_attn, next_cache = self.att(
            x,
            attn_mask,
            cos,
            sin,
            start_pos=start_pos,
            cache=cache
        )

        # Output:
        # x_attn: (B, T, D)
        # next_cache:
        #   k: (B, kv_heads, T_total, head_dim)
        #   v: (B, kv_heads, T_total, head_dim)



        # ============================================================
        # ---------------- CACHE TRUNCATION ---------------------------
        # ============================================================

        if next_cache is not None and self.attn_type == "sliding_attention":
            k, v = next_cache                          # (B, kv_heads, T_total, head_dim)

            # If cache grows beyond window → truncate
            if k.size(2) > self.sliding_window:
                # keep only last W tokens
                k = k[:, :, -self.sliding_window:, :]  # (B, kv_heads, W, head_dim)
                v = v[:, :, -self.sliding_window:, :]

            next_cache = (k, v)  ## only-hold this much cache



        # ============================================================
        # ---------------- POST-ATTENTION BLOCK -----------------------
        # ============================================================

        # RMSNorm
        x_attn = self.post_attention_layernorm(x_attn)   # (B, T, D)

        # Residual connection
        x = shortcut + x_attn                           # (B, T, D)



        # ============================================================
        # ---------------- FEEDFORWARD BLOCK --------------------------
        # ============================================================

        shortcut = x                                    # (B, T, D)

        x_fnn = self.ff(x)                              # (B, T, D)

        x_fnn = self.post_feedforward_layernorm(x_fnn)   # (B, T, D)

        # Residual
        x = shortcut + x_fnn                            # (B, T, D)



        # ============================================================
        # ---------------- FINAL OUTPUT -------------------------------
        # ============================================================

        return x, next_cache
    




In [21]:
import torch
import torch.nn as nn


class Olmo3(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # --- Sanity: one attention type per layer ---
        assert cfg["layer_types"] is not None
        assert len(cfg["layer_types"]) == cfg["n_layers"]

        # ============================================================
        # Token embedding
        # input_ids: [B, T]
        # output:    [B, T, emb_dim]
        # ============================================================
        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"],
            dtype=cfg["dtype"]
        )

        # ============================================================
        # Transformer blocks (stack)
        # Each block preserves shape: [B, T, emb_dim]
        # ============================================================
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg, attn_type)
            for attn_type in cfg["layer_types"]
        ])

        # ============================================================
        # Final normalization (RMSNorm)
        # Input:  [B, T, emb_dim]
        # Output: [B, T, emb_dim]
        # ============================================================
        self.final_norm = RMSNorm(
            cfg["emb_dim"],
            eps=cfg["rms_norm_eps"]
        )

        # ============================================================
        # Output projection → vocabulary logits
        # Input:  [B, T, emb_dim]
        # Output: [B, T, vocab_size]
        # ============================================================
        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False,
            dtype=cfg["dtype"]
        )

        self.cfg = cfg

        # ============================================================
        # Stateful decoding pointer
        # Tracks absolute position during KV-cache decoding
        # ============================================================
        self.current_pos = 0

        # ============================================================
        # Precompute RoPE tables
        # cos/sin: [max_seq_len, head_dim]
        # Stored as buffers (not trainable, move with device)
        # ============================================================
        cos, sin = compute_rope_parameters(
            head_dim=cfg["head_dim"],
            context_length=cfg["context_length"],
            theta_base=cfg["rope_base"],
            attention_factor=cfg["rope_attention_factor"],
            rope_type=cfg["rope_type"],
            rope_factor=cfg["rope_factor"],
            rope_orig_max=cfg["rope_orig_max"],
            dtype=torch.float32,
        )

        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

    # ================================================================
    # MASK CREATION
    # ================================================================
    def create_masks(self, cur_len, device, pos_start=0, pos_end=None):

        """
        cur_len:   number of current tokens (query length)
        pos_start: starting absolute position in sequence
        pos_end:   total sequence length after adding current tokens

        Returns:
            mask_global: [1, 1, cur_len, total_len]
            mask_local:  [1, 1, cur_len, total_len]
        """

        if pos_end is None:
            pos_end = cur_len

        # total sequence length (past + current)
        total_len = pos_end

        # ============================================================
        # Full square mask
        # shape: [total_len, total_len]
        # ============================================================
        ones = torch.ones(
            (total_len, total_len),
            dtype=torch.bool,
            device=device
        )

        # ============================================================
        # GLOBAL CAUSAL MASK
        # True = masked (cannot attend)
        # Upper triangular (future tokens)
        # ============================================================
        mask_global_full = torch.triu(ones, diagonal=1)

        # ============================================================
        # LOCAL MASK (sliding window constraint)
        # Masks tokens that are too far in the past
        # ============================================================
        far_past_full = torch.triu(
            ones,
            diagonal=self.cfg["sliding_window"]
        )

        # Combine:
        # - future tokens
        # - far past tokens
        mask_local_full = mask_global_full | far_past_full

        # ============================================================
        # Slice only relevant queries (rows)
        # rows   = current tokens [pos_start : pos_end]
        # cols   = all tokens     [0 : pos_end]
        # ============================================================
        row_slice = slice(pos_start, pos_end)

        # shape before unsqueeze:
        # [cur_len, total_len]
        mask_global = mask_global_full[row_slice, :pos_end]
        mask_local  = mask_local_full[row_slice, :pos_end]

        # ============================================================
        # Expand for attention:
        # [B, heads, Q, K]
        # → broadcast over batch and heads
        # ============================================================
        mask_global = mask_global[None, None, :, :]
        mask_local  = mask_local[None, None, :, :]

        return mask_global, mask_local

    # ================================================================
    # FORWARD PASS
    # ================================================================
    def forward(self, input_ids, cache=None):

        """
        input_ids: [B, T]
        cache:     dict(layer_id → KV tensors) OR None

        Returns:
            logits: [B, T, vocab_size]
        """

        b, seq_len = input_ids.shape

        # ============================================================
        # Token embedding
        # [B, T] → [B, T, emb_dim]
        # ============================================================
        x = self.tok_emb(input_ids)

        # ============================================================
        # POSITION + MASK HANDLING
        # ============================================================
        if cache is not None:
            # Inference mode (incremental decoding)

            pos_start = self.current_pos
            pos_end   = pos_start + seq_len

            # Update global pointer
            self.current_pos = pos_end

            # Masks consider full context (past + current)
            mask_global, mask_local = self.create_masks(
                cur_len=seq_len,
                device=x.device,
                pos_start=pos_start,
                pos_end=pos_end
            )

        else:
            # Training mode (full sequence)

            pos_start = 0
            pos_end   = seq_len

            mask_global, mask_local = self.create_masks(
                cur_len=seq_len,
                device=x.device,
                pos_start=pos_start,
                pos_end=pos_end
            )

        # ============================================================
        # RoPE tables (shared across layers)
        # ============================================================
        cos = self.cos   # [max_seq_len, head_dim]
        sin = self.sin   # [max_seq_len, head_dim]

        # ============================================================
        # Transformer stack
        # ============================================================
        for i, block in enumerate(self.blocks):

            # Retrieve per-layer KV cache
            blk_cache = cache.get(i) if cache is not None else None

            # Forward through block
            # x: [B, T, emb_dim]
            x, new_blk_cache = block(
                x,
                mask_global=mask_global,
                mask_local=mask_local,
                cos=cos,
                sin=sin,
                start_pos=pos_start,   # critical for RoPE alignment
                cache=blk_cache,
            )

            # Update cache (append new KV)
            if cache is not None:
                if isinstance(cache, dict):
                    cache[i] = new_blk_cache
                else:
                    cache.update(i, new_blk_cache)
        # ============================================================
        # Final normalization + projection
        # ============================================================
        x = self.final_norm(x)  # [B, T, emb_dim]

        logits = self.out_head(
            x.to(self.cfg["dtype"])
        )  # [B, T, vocab_size]

        return logits

    # ================================================================
    # RESET STATE (NEW SEQUENCE)
    # ================================================================
    def reset_kv_cache(self):
        """
        Must be called before starting a new independent sequence.
        Resets positional state (and externally, KV cache should be cleared).
        """
        self.current_pos = 0

In [22]:
"""
KV Cache: Global storage for Keys and Values across transformer layers
Used during autoregressive inference (NOT needed for full-sequence training)
"""


class KvCache:
    def __init__(self, n_layers: int):
        """
        Args:
            n_layers (int): Number of transformer layers

        Internal:
            self.cache: list of length n_layers

        Each entry:
            None
            OR
            (K, V)

        Shapes per layer:
            K: [B, H_kv, T_cached, D]
            V: [B, H_kv, T_cached, D]

        Where:
            B        = batch size
            H_kv     = number of KV heads (important for GQA)
            T_cached = total cached sequence length (grows over time)
            D        = head_dim
        """
        self.cache = [None] * n_layers


    def get(self, layer_idx: int):
        """
        Retrieve cached KV for a given layer.

        Args:
            layer_idx (int)

        Returns:
            None
            OR
            (K, V)

        Shapes:
            K: [B, H_kv, T_cached, D]
            V: [B, H_kv, T_cached, D]
        """
        return self.cache[layer_idx]


    def update(self, layer_idx: int, value):
        """
        Update KV cache for a given layer.

        Args:
            layer_idx (int)
            value (tuple): (K, V)

        Expected Shapes:
            K: [B, H_kv, T_new_total, D]
            V: [B, H_kv, T_new_total, D]

        Important:
            - This should be called AFTER concatenating past + new KV
            - Overwrites previous cache with extended sequence

        Example (outside this class):
            past_k, past_v = kv_cache.get(layer_idx)

            if past_k is not None:
                K = torch.cat([past_k, new_k], dim=2)
                V = torch.cat([past_v, new_v], dim=2)
            else:
                K, V = new_k, new_v

            kv_cache.update(layer_idx, (K, V))
        """
        self.cache[layer_idx] = value


    def get_all(self):
        """
        Returns full KV cache across all layers.

        Returns:
            List of length n_layers:
                [
                    (K_0, V_0),
                    (K_1, V_1),
                    ...
                ]

        Shapes:
            Each:
                K_l: [B, H_kv, T_cached, D]
                V_l: [B, H_kv, T_cached, D]
        """
        return self.cache


    def reset(self):
        """
        Clears the cache (for new sequence).

        After reset:
            All entries → None

        Important:
            Must be called between independent generations
            to avoid leakage across prompts.
        """
        for i in range(len(self.cache)):
            self.cache[i] = None

In [23]:
import torch

# ============================================================
# Olmo3 Configuration (7B)
# ============================================================

OLMO3_CONFIG_7B = {
    # -----------------------------
    # Token / sequence space
    # -----------------------------
    "vocab_size": 100_278,        # |V| → embedding table: [vocab_size, emb_dim]
    "context_length": 65_536,     # max sequence length L

    # -----------------------------
    # Model dimensions
    # -----------------------------
    "emb_dim": 4_096,             # model width d_model
                                  # token embedding: [B, L] → [B, L, 4096]

    "n_heads": 32,                # total attention heads H
    "head_dim": 128,              # per-head dim d_h
                                  # invariant: emb_dim = n_heads * head_dim

    "n_kv_heads": 32,             # KV heads (GQA)
                                  # here: full MHA (no grouping)

    "n_layers": 32,               # transformer depth

    "hidden_dim": 11_008,         # FFN intermediate dim
                                  # MLP: [B, L, 4096] → [B, L, 11008] → [B, L, 4096]

    # -----------------------------
    # Attention behavior
    # -----------------------------
    "attention_bias": False,      # no bias in QKV projections
    "attention_dropout": 0.0,     # typically 0 for inference / post-training

    "sliding_window": 4_096,      # local attention window size W
                                  # sliding mask: each token attends to last W tokens

    "layer_types": [
        # Pattern: 3x sliding + 1x full attention (global refresh)
        # total = 32 layers
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
    ],
    # sliding_attention:
    #   attention mask → [B, H, L, L] (banded)
    # full_attention:
    #   attention mask → [B, H, L, L] (causal full)

    # -----------------------------
    # RoPE (YaRN scaling)
    # -----------------------------
    "rope_base": 500_000.0,
    "rope_type": "yarn",

    "rope_factor": 8.0,           # context extension factor
    "rope_orig_max": 8_192,       # original trained context

    "rope_attention_factor": 1.2079441541679836,

    "beta_fast": 32.0,            # YaRN interpolation params
    "beta_slow": 1.0,

    # RoPE applied to:
    # Q, K: [B, H, L, d_h] → rotated along last dim

    # -----------------------------
    # Normalization
    # -----------------------------
    "rms_norm_eps": 1e-6,         # RMSNorm stability

    # -----------------------------
    # Precision / tokens
    # -----------------------------
    "dtype": torch.bfloat16,      # compute + weights dtype

    "eos_token_id": 100_257,      # end-of-sequence token
    "pad_token_id": 100_277,      # padding token
}


# ============================================================
# Olmo3 Configuration (32B)
# ============================================================

OLMO3_CONFIG_32B = {
    # -----------------------------
    # Token / sequence space
    # -----------------------------
    "vocab_size": 100_278,
    "context_length": 65_536,

    # -----------------------------
    # Model dimensions
    # -----------------------------
    "emb_dim": 5_120,             # [B, L, 5120]

    "n_heads": 40,
    "head_dim": 128,              # 40 * 128 = 5120

    "n_kv_heads": 8,              # GQA (grouped-query attention)
                                  # queries: 40 heads
                                  # keys/values: 8 heads
                                  # group_size = 40 / 8 = 5

    "n_layers": 64,

    "hidden_dim": 27_648,         # large FFN expansion

    # -----------------------------
    # Attention behavior
    # -----------------------------
    "attention_bias": False,
    "attention_dropout": 0.0,

    "sliding_window": 4_096,

    "layer_types": [
        # repeating pattern across 64 layers
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
        "sliding_attention", "sliding_attention", "sliding_attention", "full_attention",
    ],

    # -----------------------------
    # RoPE (YaRN scaling)
    # -----------------------------
    "rope_base": 500_000.0,
    "rope_type": "yarn",

    "rope_factor": 8.0,
    "rope_orig_max": 8_192,

    "rope_attention_factor": 1.2079441541679836,

    "beta_fast": 32.0,
    "beta_slow": 1.0,

    # -----------------------------
    # Normalization
    # -----------------------------
    "rms_norm_eps": 1e-6,

    # -----------------------------
    # Precision / tokens
    # -----------------------------
    "dtype": torch.bfloat16,

    "eos_token_id": 100_257,
    "pad_token_id": 100_277,
}


# ============================================================
# Model selection
# ============================================================

# USE_MODEL: string flag (e.g., "7B" or "32B")
OLMO3_CONFIG = (
    OLMO3_CONFIG_32B if "32" in USE_MODEL else OLMO3_CONFIG_7B
)

In [24]:
torch.manual_seed(123)

## Model here
model = Olmo3(OLMO3_CONFIG)


In [26]:
model

Olmo3(
  (tok_emb): Embedding(100278, 4096)
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (att): GroupQueryAttention(
        (W_query): Linear(in_features=4096, out_features=4096, bias=False)
        (W_key): Linear(in_features=4096, out_features=4096, bias=False)
        (W_value): Linear(in_features=4096, out_features=4096, bias=False)
        (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=4096, out_features=11008, bias=False)
        (fc2): Linear(in_features=4096, out_features=11008, bias=False)
        (fc3): Linear(in_features=11008, out_features=4096, bias=False)
      )
      (post_attention_layernorm): RMSNorm()
      (post_feedforward_layernorm): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=4096, out_features=100278, bias=False)
)

In [27]:
model(torch.tensor([1,2,3]).unsqueeze(0))

tensor([[[ 0.3691, -0.6367, -0.2754,  ...,  1.1406,  0.4355,  0.0239],
         [ 1.2891,  0.0040,  0.4863,  ...,  0.5742, -0.2471,  0.1748],
         [ 0.6016, -0.0334,  0.7773,  ...,  0.3184, -0.5312, -0.1758]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

In [ ]:
# ============================================================
# Device selection (compute backend)
# ============================================================

# Goal:
# Select the fastest available device for tensor computation.
# Priority: CUDA (NVIDIA GPU) > MPS (Apple GPU) > CPU

if torch.cuda.is_available():
    # CUDA backend → NVIDIA GPU
    # Enables high throughput (tensor cores, bf16/fp16 support)
    device = torch.device("cuda")

elif torch.backends.mps.is_available():
    # MPS backend → Apple Silicon GPU (M1/M2/M3)
    # Uses Metal Performance Shaders
    device = torch.device("mps")

else:
    # Fallback → CPU (slowest, but always available)
    device = torch.device("cpu")


# ============================================================
# Move model to selected device
# ============================================================

# This moves:
# - all model parameters (weights)
# - buffers (e.g., layer norm stats, KV cache if registered)
# from CPU → selected device (GPU/MPS/CPU)

model.to(device)


# ============================================================
# IMPORTANT INVARIANT
# ============================================================

# All inputs MUST be on the same device as the model
# Example:
#   x = x.to(device)
#   out = model(x)
#
# Otherwise → runtime error:
# "Expected all tensors to be on the same device"

## Load Pre-trained weights

In [ ]:
## load_weights function 
def load_weights_into_olmo(model, param_config, params):

    def assign(left, right, tensor_name="unknown"):
        """
        Copy pretrained weight (right) → model parameter (left)

        left  : torch.nn.Parameter (destination, already part of model)
        right : tensor / array from checkpoint

        Ensures:
        - same shape
        - correct dtype + device
        - no gradient tracking
        """
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. "
                f"Left: {left.shape}, Right: {right.shape}"
            )
        
        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)  # in-place copy
            else: 
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left


    ## =========================
    ## Token Embedding
    ## =========================
    # Shape: [vocab_size, d_model]
    # Maps token_id → vector representation
    if "model.embed_tokens.weight" in params:  
        model.tok_emb.weight = assign(
            model.tok_emb.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight",
        )


    ## =========================
    ## Transformer Blocks
    ## =========================
    for l in range(param_config["n_layers"]):
        block = model.blocks[l]
        att = block.att


        ## ---- Q, K, V projections ----
        # All shapes: [d_model, d_model]
        # Used as: Q = X @ W_query^T (PyTorch Linear uses transpose internally)

        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )

        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )

        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )


        ## ---- Output projection ----
        # Shape: [d_model, d_model]
        # Combines multi-head outputs back to model dimension
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],  # fixed typo
            f"model.layers.{l}.self_attn.o_proj.weight",
        )


        ## ---- Q / K normalization ----
        # Shape: [d_model]  (RMSNorm-style scaling)
        # Applied AFTER projection, BEFORE attention scores
        att.q_norm.weight = assign(
            att.q_norm.weight,
            params[f"model.layers.{l}.self_attn.q_norm.weight"],
            f"model.layers.{l}.self_attn.q_norm.weight",
        )

        att.k_norm.weight = assign(
            att.k_norm.weight,
            params[f"model.layers.{l}.self_attn.k_norm.weight"],
            f"model.layers.{l}.self_attn.k_norm.weight",
        )


        ## =========================
        ## Feedforward (MLP)
        ## =========================
        # Typical gated MLP (SwiGLU-style)

        # fc1 (gate_proj): [d_model, d_ff]
        # produces gating values
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )

        # fc2 (up_proj): [d_model, d_ff]
        # expands representation
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )

        # fc3 (down_proj): [d_ff, d_model]
        # projects back to model dimension
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )


        ## =========================
        ## LayerNorms (Post)
        ## =========================

        # Shape: [d_model]
        # Applied AFTER attention residual
        block.post_attention_layernorm.weight = assign(
            block.post_attention_layernorm.weight,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight",
        )

        # Shape: [d_model]
        # Applied AFTER FFN residual
        block.post_feedforward_layernorm.weight = assign(
            block.post_feedforward_layernorm.weight,
            params[f"model.layers.{l}.post_feedforward_layernorm.weight"],
            f"model.layers.{l}.post_feedforward_layernorm.weight",
        )


    ## =========================
    ## Final Normalization
    ## =========================
    # Shape: [d_model]
    # Applied after all transformer blocks
    if "model.norm.weight" in params:
        model.final_norm.weight = assign(
            model.final_norm.weight,
            params["model.norm.weight"],
            "model.norm.weight",
        )


    ## =========================
    ## Output Head (Logits)
    ## =========================
    # Shape: [vocab_size, d_model]
    # Used as: logits = h @ W_out^T → [B, T, vocab]

    if "lm_head.weight" in params:
        # separate output layer
        model.out_head.weight = assign(
            model.out_head.weight,
            params["lm_head.weight"],
            "lm_head.weight",
        )
    else:
        # weight tying: reuse embedding matrix
        # logits = h @ E^T
        model.out_head.weight = model.tok_emb.weight  # fixed typo
        print("Model uses weight tying.")

In [ ]:
##
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import snapshot_download


## ------------------------------------------------------------
## Repo identification
## ------------------------------------------------------------
USE_MODEL = "7b"

# Hugging Face repo name
repo_id = f"allenai/{USE_MODEL}"          # e.g. "allenai/7b"

# Extract last part → local folder name
local_dir = Path(repo_id).parts[-1]      # "7b"


## ------------------------------------------------------------
## Download full model snapshot locally
## ------------------------------------------------------------
# repo_dir: str → path to downloaded directory
# Example: "./7b/"
repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)


## ------------------------------------------------------------
## Load index file (routing table)
## ------------------------------------------------------------
# Path to index file describing sharded weights
index_path = os.path.join(repo_dir, "model.safetensors.index.json")

with open(index_path, "r") as f:
    index = json.load(f)

# index: dict
# {
#   "weight_map": {
#       "model.layers.0.attn.q_proj.weight": "model-00001-of-00005.safetensors",
#       ...
#   }
# }


## ------------------------------------------------------------
## Load all shards and merge into one dictionary
## ------------------------------------------------------------
weights_dict = {}
# weights_dict: Dict[str, torch.Tensor]
# FINAL STRUCTURE:
# {
#   "model.embed_tokens.weight":        (vocab_size, hidden_dim)
#   "model.layers.0.attn.q_proj.weight": (hidden_dim, hidden_dim)
#   ...
# }

# Extract unique shard filenames
for filename in sorted(set(index["weight_map"].values())):
    # filename: str
    # e.g. "model-00001-of-00005.safetensors"

    # Full path to shard file
    shard_path = os.path.join(repo_dir, filename)

    # Load one shard
    shard = load_file(shard_path)
    # shard: Dict[str, torch.Tensor]
    # Example:
    # {
    #   "model.embed_tokens.weight": Tensor[32000, 4096]
    #   "model.layers.0.attn.q_proj.weight": Tensor[4096, 4096]
    #   ...
    # }

    # Merge into global dictionary
    weights_dict.update(shard)
    # After multiple iterations:
    # weights_dict contains ALL tensors from ALL shards


## ------------------------------------------------------------
## Example tensor shapes inside weights_dict (OLMo-style)
## ------------------------------------------------------------
# weights_dict["model.embed_tokens.weight"]           → (vocab_size, hidden_dim)
# weights_dict["model.layers.0.attn.q_proj.weight"]   → (hidden_dim, hidden_dim)
# weights_dict["model.layers.0.attn.k_proj.weight"]   → (hidden_dim, hidden_dim)
# weights_dict["model.layers.0.attn.v_proj.weight"]   → (hidden_dim, hidden_dim)
# weights_dict["model.layers.0.attn.o_proj.weight"]   → (hidden_dim, hidden_dim)
# weights_dict["model.layers.0.mlp.up_proj.weight"]   → (hidden_dim, 4 * hidden_dim)
# weights_dict["model.layers.0.mlp.down_proj.weight"] → (4 * hidden_dim, hidden_dim)
# weights_dict["model.layers.0.input_layernorm.weight"] → (hidden_dim,)
# weights_dict["model.norm.weight"]                   → (hidden_dim,)
# weights_dict["lm_head.weight"]                      → (vocab_size, hidden_dim)


## ------------------------------------------------------------
## Load weights into your custom OLMo model
## ------------------------------------------------------------
# This function maps:
# weights_dict keys → model parameter tensors
load_weights_into_olmo(model, OLMO3_CONFIG, weights_dict)


## ------------------------------------------------------------
## Move model to device (GPU / CPU / MPS)
## ------------------------------------------------------------
model.to(device)


## ------------------------------------------------------------
## Free CPU RAM (important for large models)
## ------------------------------------------------------------
del weights_dict

### load_tokenizer

In [ ]:
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download
from pathlib import Path
import os


class OlmoTokenizer:
    def __init__(self, tokenizer_file_path, eos_token_id, pad_token_id):
        """
        tokenizer_file_path : str → path to tokenizer.json
        eos_token_id        : int → fallback EOS id
        pad_token_id        : int → fallback PAD id
        """

        ## ------------------------------------------------------------
        ## Load tokenizer (vocab + merges + rules)
        ## ------------------------------------------------------------
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        # self._tok: HuggingFace Tokenizer object


        ## ------------------------------------------------------------
        ## Resolve EOS token (End Of Sequence)
        ## ------------------------------------------------------------
        eos_from_tok = (
            self._tok.token_to_id("<|endoftext|>")   # GPT-style EOS
            or self._tok.token_to_id("<end_of_turn>")  # chat-style EOS
        )
        # eos_from_tok: int OR None

        # Final EOS:
        # use tokenizer value if available, else fallback
        self.eos_token_id = (
            eos_from_tok if eos_from_tok is not None else eos_token_id
        )
        # self.eos_token_id: int


        ## ------------------------------------------------------------
        ## Resolve PAD token (for batching + masking)
        ## ------------------------------------------------------------
        pad_from_tok = (
            self._tok.token_to_id("<|pad|>")   # common pad
            or self._tok.token_to_id("<pad>")  # alternative pad
        )
        # pad_from_tok: int OR None

        # Final PAD:
        self.pad_token_id = (
            pad_from_tok if pad_from_tok is not None else pad_token_id
        )
        # self.pad_token_id: int


    ## ------------------------------------------------------------
    ## Encode: text → token IDs
    ## ------------------------------------------------------------
    def encode(self, text):
        """
        text : str

        Returns:
            ids : List[int]  shape = (seq_len,)
        """
        # Example:
        # "Hello" → [15496, 995]
        return self._tok.encode(text).ids


    ## ------------------------------------------------------------
    ## Decode: token IDs → text
    ## ------------------------------------------------------------
    def decode(self, ids):
        """
        ids : List[int] shape = (seq_len,)

        Returns:
            text : str
        """
        # skip_special_tokens=False → keeps EOS, PAD, etc.
        return self._tok.decode(ids, skip_special_tokens=False)


    ## ------------------------------------------------------------
    ## Chat template: formats input for chat models
    ## ------------------------------------------------------------
    @staticmethod  
    def apply_chat_template(user_text):
        """
        user_text : str

        Returns:
            formatted_text : str

        Example output:
        <|im_start|>user
        Hello
        <|im_end|>
        <|im_start|>assistant
        """
        return (
            "<|im_start|>user\n"
            f"{user_text}\n"
            "<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        # shape: string (later becomes tokens → (seq_len,))


## ============================================================
## Tokenizer file handling (download if missing)
## ============================================================

# Path to tokenizer file
tokenizer_file_path = os.path.join(local_dir, "tokenizer.json")

if not os.path.exists(tokenizer_file_path):
    try:
        # Download tokenizer.json from HF repo
        hf_hub_download(
            repo_id=repo_id,
            filename="tokenizer.json",
            local_dir=local_dir
        )
    except Exception as e:
        print(f"failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"  # fallback path


## ============================================================
## Initialize tokenizer
## ============================================================

tokenizer = OlmoTokenizer(
    tokenizer_file_path=tokenizer_file_path,
    eos_token_id=OLMO3_CONFIG["eos_token_id"],  # int
    pad_token_id=OLMO3_CONFIG["pad_token_id"]   # int
)




c:\Users\mailm\Documents\Olmo-end-to-end-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'local_dir' is not defined

In [ ]:
## ------------------------------------------------------------
## Step 1: Apply chat template (text formatting)
## ------------------------------------------------------------

prompt = OlmoTokenizer.apply_chat_template(
    "What is a neural network in simple words"
)

# prompt: str
# Example:
# "<|im_start|>user\nWhat is a neural network in simple words\n<|im_end|>\n<|im_start|>assistant\n"


## ------------------------------------------------------------
## Step 2: Encode (text → token IDs)
## ------------------------------------------------------------

input_token_ids = tokenizer.encode(prompt)

# input_token_ids: List[int]
# shape = (seq_len,)
#
# Example:
# [101, 345, 8921, ..., 50256]
#
# seq_len depends on:
# - length of text
# - tokenizer (BPE splits, etc.)


## ------------------------------------------------------------
## Step 3: Decode (token IDs → text)
## ------------------------------------------------------------

text = tokenizer.decode(input_token_ids)

# text: str
# Should reconstruct original prompt (including special tokens)
#
# Because:
# skip_special_tokens = False


## ------------------------------------------------------------
## Final output
## ------------------------------------------------------------

text
# str (same as prompt, unless tokenizer normalizes spacing)

### generate text

In [ ]:
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None, context_size=None):
    """
    model          : causal LM
    token_ids      : torch.Tensor  shape = (B, T)
                     B = batch size, T = current sequence length
    max_new_tokens : int
    eos_token_id   : int or None
    """

    ## ------------------------------------------------------------
    ## Set model to inference mode (disable dropout, etc.)
    ## ------------------------------------------------------------
    model.eval()

    with torch.no_grad():

        ## ------------------------------------------------------------
        ## Initialize KV cache (stores past keys/values per layer)
        ## ------------------------------------------------------------
        cache = KvCache(n_layers=model.cfg["n_layers"])
        # cache: stores tensors of shape ~ (B, H, T, D_head) per layer

        model.reset_kv_cache()  # ensure clean state


        ## ------------------------------------------------------------
        ## First forward pass (full prompt)
        ## ------------------------------------------------------------
        logits = model(token_ids, cache=cache)
        # token_ids: (B, T)
        # logits   : (B, T, V)
        # V = vocab size


        ## ------------------------------------------------------------
        ## Autoregressive generation loop
        ## ------------------------------------------------------------
        for _ in range(max_new_tokens):

            ## --------------------------------------------------------
            ## Step 1: pick next token from last position
            ## --------------------------------------------------------
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)

            # logits[:, -1] → (B, V)  → last position
            # argmax        → (B,)
            # keepdim=True  → (B, 1)

            # next_token: (B, 1)
            # contains vocab indices (token IDs)


            ## --------------------------------------------------------
            ## Step 2: check EOS condition (stop if all sequences ended)
            ## --------------------------------------------------------
            if (
                eos_token_id is not None
                and torch.all(next_token == eos_token_id)
            ):
                # torch.all(...) → scalar bool
                # True if ALL batch sequences produced EOS
                break


            ## --------------------------------------------------------
            ## Step 3: stream output token
            ## --------------------------------------------------------
            yield next_token
            # next_token: (B, 1)
            # returned one token at a time (streaming)


            ## --------------------------------------------------------
            ## Step 4: append new token to sequence
            ## --------------------------------------------------------
            token_ids = torch.cat([token_ids, next_token], dim=1)

            # before: token_ids → (B, T)
            # next_token        → (B, 1)
            # after: token_ids → (B, T+1)


            ## --------------------------------------------------------
            ## Step 5: next forward pass (KV cache optimization)
            ## --------------------------------------------------------
            logits = model(next_token, cache=cache)

            # input: next_token → (B, 1)
            # cache: contains past K/V (no recomputation)
            # output:
            # logits → (B, 1, V)

            # NOTE:
            # Only new token is processed, past tokens reused via cache

In [ ]:
## ------------------------------------------------------------
## Step 1: Convert token list → tensor + add batch dimension
## ------------------------------------------------------------

input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

# input_token_ids        : List[int]              shape = (T,)
# torch.tensor(...)      : torch.Tensor           shape = (T,)
# .unsqueeze(0)          : add batch dimension → (1, T)


# Final:
# input_token_ids_tensor : torch.Tensor           shape = (B, T)
# where B = 1, T = sequence length


## ------------------------------------------------------------
## Step 2: Reset GPU memory stats (for profiling)
## ------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

    # Clears previous peak memory tracking
    # so we measure ONLY this generation run


## ------------------------------------------------------------
## Step 3: Streaming generation loop
## ------------------------------------------------------------

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,   # (B, T)
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):

    # token: torch.Tensor   shape = (B, 1)
    # Example: tensor([[50256]])


    ## --------------------------------------------------------
    ## Step 3.1: Remove batch dimension + convert to Python list
    ## --------------------------------------------------------

    token_id = token.squeeze(0).tolist()

    # token.squeeze(0): (1, 1) → (1,)
    # .tolist():       (1,) → [token_id]

    # token_id: List[int]   shape = (1,)


    ## --------------------------------------------------------
    ## Step 3.2: Decode and print token (streaming output)
    ## --------------------------------------------------------

    print(
        tokenizer.decode(token_id),
        end="",      # no newline → continuous text
        flush=True   # print immediately (real-time streaming)
    )

    # tokenizer.decode([id]) → str
    # prints one token at a time


## ------------------------------------------------------------
## Step 4: Report GPU memory usage (if using CUDA)
## ------------------------------------------------------------

if torch.cuda.is_available():

    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    # Converts bytes → GB

    print(
        f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}"
    )

    # torch.cuda.max_memory_allocated():
    # returns peak GPU memory (in bytes) used during this run